In [1]:
import os
import sys
from pathlib import Path

### Locate project root dynamically by searching upwards for 'hybrid' or 'data'

In [2]:
current = Path.cwd()
while current != current.parent:
    if (current / "hybrid").exists() and (current / "data").exists():
        PROJECT_ROOT = current
        break
    current = current.parent
else:
    # Hardcoded fallback if directory tree search fails
    PROJECT_ROOT = Path("/Users/rahulkpkurup/Learning/ML-Git-Portfolio/ncf-recommender-system-study").resolve()

# Change working directory and update python path
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hybrid.hybrid_engine import HybridEngine

meta_path = PROJECT_ROOT / "data" / "processed" / "meta.json"
weights_path = PROJECT_ROOT / "evaluation" / "pytorch_ncf_weights.pt"

engine = HybridEngine.load(weights_path=weights_path, meta_path=meta_path)
print("Engine loaded successfully from:", PROJECT_ROOT)

Engine loaded successfully from: /Users/rahulkpkurup/Learning/ML-Git-Portfolio/ncf-recommender-system-study


### Test 1: cold-start user (0 interactions)

In [3]:
cold_recs = engine.recommend(user_id=0, seen_items=[], k=5)
print("Cold-start recommendations:")
for r in cold_recs:
    print(f"  item {r['item_id']:4d} | score {r['score']:.3f} | "
          f"source: {r['source']} | alpha: {r['alpha']}")

Cold-start recommendations:
  item 2651 | score 1.000 | source: cold_start | alpha: 0.0
  item  253 | score 0.835 | source: cold_start | alpha: 0.0
  item 1106 | score 0.823 | source: cold_start | alpha: 0.0
  item 1120 | score 0.804 | source: cold_start | alpha: 0.0
  item 1848 | score 0.729 | source: cold_start | alpha: 0.0


### Test 2: warm user (simulate 15 interactions)

In [4]:
seen = [1, 5, 23, 50, 88, 120, 200, 310, 450, 600, 700, 800, 900, 1000, 1100]
warm_recs = engine.recommend(user_id=0, seen_items=seen, k=5)
print("\nWarm user recommendations:")
for r in warm_recs:
    print(f"  item {r['item_id']:4d} | score {r['score']:.3f} | "
          f"ncf: {r['ncf_score']:.3f} | cold: {r['cold_score']:.3f} | "
          f"alpha: {r['alpha']}")


Warm user recommendations:
  item 2426 | score 0.984 | ncf: 0.992 | cold: 0.872 | alpha: 0.75
  item 1107 | score 0.976 | ncf: 0.995 | cold: 0.857 | alpha: 0.75
  item  574 | score 0.962 | ncf: 0.983 | cold: 0.850 | alpha: 0.75
  item  106 | score 0.948 | ncf: 0.938 | cold: 0.877 | alpha: 0.75
  item 2599 | score 0.945 | ncf: 0.975 | cold: 0.835 | alpha: 0.75


### Test 3: fully warm user (20+ interactions)

In [5]:
full_seen = list(range(1, 25))
full_recs = engine.recommend(user_id=1, seen_items=full_seen, k=5)
print("\nFully warm recommendations (NCF dominant):")
for r in full_recs:
    print(f"  item {r['item_id']:4d} | score {r['score']:.3f} | "
          f"source: {r['source']}")


Fully warm recommendations (NCF dominant):
  item  339 | score 1.000 | source: ncf
  item 3032 | score 0.999 | source: ncf
  item  737 | score 0.995 | source: ncf
  item 3186 | score 0.995 | source: ncf
  item 1271 | score 0.994 | source: ncf


### Test 4: similar item (search fallback)

In [6]:
similar = engine.recommend_similar(item_id=1, k=5)
print("\nItems similar to item 1:")
for r in similar:
    print(f"  item {r['item_id']:4d} | similarity: {r['score']:.3f}")


Items similar to item 1:
  item 1064 | similarity: 1.000
  item 2141 | similarity: 1.000
  item 2142 | similarity: 1.000
  item 2354 | similarity: 1.000
  item 2355 | similarity: 1.000


### Test 4: similar item (search fallback)

In [7]:
similar = engine.recommend_similar(item_id=1, k=5)
print("\nItems similar to item 1:")
for r in similar:
    print(f"  item {r['item_id']:4d} | similarity: {r['score']:.3f}")


Items similar to item 1:
  item 1064 | similarity: 1.000
  item 2141 | similarity: 1.000
  item 2142 | similarity: 1.000
  item 2354 | similarity: 1.000
  item 2355 | similarity: 1.000


### Test 5A: Direct check - Content-based User Profile generation

In [9]:
test_seen = [1, 5, 23]
user_profile = engine.content.user_profile(test_seen)
print(f"User Profile generated: {user_profile is not None}")

User Profile generated: True


### Test 5B: Direct check - Content similarity score for candidate items

In [10]:
candidate_item_id = 50
content_score = engine.content.score(user_profile, candidate_item_id)
print(f"Content score for item {candidate_item_id}: {content_score:.4f}")

Content score for item 50: 0.2673


### Test 5C: Direct check - Compare Content-Only vs Popularity-Only on cold-start

In [11]:
candidates = engine.pop.top_k(10, exclude=set(test_seen))
print("\nDirect Cold-Start Breakdown:")
for iid in candidates[:5]:
    p_score = engine.pop.score(iid)
    c_score = engine.content.score(user_profile, iid)
    print(f"  Item {iid:4d} | Pop Score: {p_score:.3f} | Content Score: {c_score:.3f}")


Direct Cold-Start Breakdown:
  Item 2651 | Pop Score: 1.000 | Content Score: 0.000
  Item  253 | Pop Score: 0.985 | Content Score: 0.000
  Item 1106 | Pop Score: 0.984 | Content Score: 0.000
  Item 1120 | Pop Score: 0.982 | Content Score: 0.000
  Item 1848 | Pop Score: 0.975 | Content Score: 0.567
